In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
import os
import pandas as pd
from collections import defaultdict
import csv
import json

from modules.utils import clean_url


In [22]:
# --------------------------------------------------------------------------------
# 1. Read in error logs from main error-logging folder (ignoring InsecureCertificateError)
#    and from http_recrawl/error-logging (keeping InsecureCertificateError).
# --------------------------------------------------------------------------------

# Paths to error log folders
MAIN_ERROR_PATH = "../analysis_data/error-logging"
RECRAWL_ERROR_PATH = "../analysis_data/http_recrawl/error-logging"

# We will store errors in these sets:
#   country_errors[country] = set of cleaned URLs that encountered ANY error 
#                            (based on ignoring or keeping InsecureCertificateError).
#   all_failed_urls = set of all cleaned URLs that encountered any error 
#                     in any country (subject to the ignoring/keeping rules).
country_errors = defaultdict(set)
all_failed_urls = set()
url_countries_map = defaultdict(set)

def extract_country_name_from_error_filename(fname):
    """
    Given an error log filename like:
      - error-logging_australia_05.json
      - error-logging_australia_re.json
    Extract 'australia' as the country name.
    """
    base = fname.replace("error-logging_", "").replace(".json", "")
    # base could be something like 'australia_05' or 'australia_re'
    # The first part is typically the country
    parts = base.split("_")
    # 'parts[0]' would be the country name (e.g., 'australia')
    return parts[0]

def read_error_logs(folder_path, skip_insecure_certificate=True):
    """
    Reads .json files from 'folder_path'.
    
    :param skip_insecure_certificate: 
        If True, do NOT include 'InsecureCertificateError' in errors.
        If False, include it.
    """
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            country = extract_country_name_from_error_filename(file_name)
            file_path = os.path.join(folder_path, file_name)
            
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    
                    for error_type, urls in data.items():
                        if skip_insecure_certificate and error_type == "InsecureCertificateError":
                            # In the MAIN logs, we ignore these
                            continue
                        # Otherwise, add to our sets
                        for url in urls:
                            c_url = clean_url(url)
                            country_errors[country].add(c_url)
                            all_failed_urls.add(c_url)
                            url_countries_map[c_url].add(country)
            except Exception as e:
                print(f"Error reading {file_name}: {e}")

# Read from main error logs, ignoring InsecureCertificateError
read_error_logs(MAIN_ERROR_PATH, skip_insecure_certificate=True)

# Read from recrawl error logs, keeping InsecureCertificateError
read_error_logs(RECRAWL_ERROR_PATH, skip_insecure_certificate=False)

# Save an intermediate file of all failing URLs and the countries that reported them

intermediate_errors_file = os.path.join(MAIN_ERROR_PATH, "all_error_sites_across_countries.csv")

if os.path.exists(intermediate_errors_file):
    os.remove(intermediate_errors_file)

with open(intermediate_errors_file, "w", encoding="utf-8", newline="") as f:
    f.write("URL,Countries\n")
    for url in sorted(url_countries_map.keys()):
        countries_list = sorted(url_countries_map[url])
        line = f"{url},\"{';'.join(countries_list)}\"\n"
        f.write(line)
print(f"Intermediate file with error sites saved to: {intermediate_errors_file}")

Intermediate file with error sites saved to: ../analysis_data/error-logging\all_error_sites_across_countries.csv


In [23]:
# --------------------------------------------------------------------------------
# 2. Combine all entries from ../analysis_data/entries AND ../analysis_data/http_recrawl/entries
# --------------------------------------------------------------------------------

MAIN_ENTRIES_PATH = "../analysis_data/entries"
RECRAWL_ENTRIES_PATH = "../analysis_data/http_recrawl/entries"

# remove "combined_all_countries.csv" in MAIN_ENTRIES_PATH
if "combined_all_countries.csv" in os.listdir(MAIN_ENTRIES_PATH):
    os.remove(os.path.join(MAIN_ENTRIES_PATH, "combined_all_countries.csv"))

all_entries = []  # we'll collect DataFrames here

def parse_country_from_entries_filename(fname):
    """
    For filenames like:
      - entries_australia_05.csv
      - entries_brazil_06.csv
      - entries_australia_re.csv
    Return 'australia' or 'brazil', ignoring the trailing '_re' or the numeric part.
    """
    # Remove prefix 'entries_' and suffix '.csv'
    base = fname.replace("entries_", "").replace(".csv", "")
    # base might be 'australia_05', 'brazil_06', or 'australia_re'
    parts = base.split("_")
    # the first chunk is the country
    country = parts[0]
    return country

def is_recrawl_file(fname):
    """
    Check if the filename indicates it's from the http_recrawl (i.e. ends with '_re.csv') 
    or if it's simply in the recrawl folder.
    """
    return fname.endswith("_re.csv")

# Read from main entries folder
for fname in os.listdir(MAIN_ENTRIES_PATH):
    if fname.endswith(".csv"):
        fpath = os.path.join(MAIN_ENTRIES_PATH, fname)
        try:
            df_temp = pd.read_csv(fpath, dtype=str)  # read everything as string to avoid type issues
            # parse country
            country = parse_country_from_entries_filename(fname)
            df_temp.insert(0, "country", country)  # Insert "country" as first column
            # Mark if it's a recrawl file
            df_temp["httpRecrawl"] = False  # Not in recrawl folder
            all_entries.append(df_temp)
        except Exception as e:
            print(f"Error reading {fpath}: {e}")

# Read from recrawl entries folder
for fname in os.listdir(RECRAWL_ENTRIES_PATH):
    if fname.endswith(".csv"):
        fpath = os.path.join(RECRAWL_ENTRIES_PATH, fname)
        try:
            df_temp = pd.read_csv(fpath, dtype=str)
            # parse country
            country = parse_country_from_entries_filename(fname)
            df_temp.insert(0, "country", country)
            # Mark if it's a recrawl file
            df_temp["httpRecrawl"] = True
            all_entries.append(df_temp)
        except Exception as e:
            print(f"Error reading {fpath}: {e}")

# Combine all into one big DataFrame
df_all = pd.concat(all_entries, ignore_index=True)

Error reading ../analysis_data/entries\crawl_time_stats.csv: cannot insert country, already exists
Error reading ../analysis_data/entries\filtered_germany_location.csv: cannot insert country, already exists
Error reading ../analysis_data/entries\filtered_singapore_location.csv: cannot insert country, already exists
Error reading ../analysis_data/entries\filtered_southkorea_location.csv: cannot insert country, already exists
Error reading ../analysis_data/entries\filtered_spain_location.csv: cannot insert country, already exists
Error reading ../analysis_data/entries\filtered_unitedstates_location.csv: cannot insert country, already exists
Error reading ../analysis_data/entries\southkorea_http_filtered.csv: cannot insert country, already exists


In [24]:
# --------------------------------------------------------------------------------
# 2.1 Read country-specific entries and mark them using "timestp", "id", and "rootUrl" matching for each country
# --------------------------------------------------------------------------------

COUNTRY_SPECIFIC_ENTRIES_PATH = "../analysis_data/country_specific_entries"
COUNTRY_SPECIFIC_RECRAWL_PATH = "../analysis_data/country_specific_entries/http_recrawl"

# Dictionary mapping each country to a set of tuples (timestp, id, cleaned_rootUrl)
country_specific_entries_set = defaultdict(set)

def parse_country_from_entries_filename(fname):
    """
    For filenames like:
      - entries_australia_05.csv
      - entries_brazil_re.csv
    Return the country (e.g., 'australia' or 'brazil').
    """
    base = fname.replace("entries_", "").replace(".csv", "")
    parts = base.split("_")
    return parts[0]

def read_country_specific_entries(folder_path):
    for fname in os.listdir(folder_path):
        if fname.endswith(".csv"):
            fpath = os.path.join(folder_path, fname)
            try:
                df_temp = pd.read_csv(fpath, dtype=str)
                country = parse_country_from_entries_filename(fname)
                required_columns = ['timestp', 'id', 'rootUrl']
                if not all(col in df_temp.columns for col in required_columns):
                    print(f"Missing required columns in {fpath}. Skipping.")
                    continue
                # Convert timestp to numeric and drop any rows with invalid timestamps
                df_temp['timestp'] = pd.to_numeric(df_temp['timestp'], errors='coerce')
                df_temp = df_temp.dropna(subset=['timestp'])
                # Iterate over each row to collect the tuple (timestp, id, cleaned rootUrl)
                for _, row in df_temp.iterrows():
                    ts = int(row['timestp'])
                    entry_id = row['id']
                    cleaned_rooturl = clean_url(row['rootUrl'])
                    country_specific_entries_set[country].add((ts, entry_id, cleaned_rooturl))
            except Exception as e:
                print(f"Error reading {fpath}: {e}")

# Read entries from both country-specific folders
read_country_specific_entries(COUNTRY_SPECIFIC_ENTRIES_PATH)
read_country_specific_entries(COUNTRY_SPECIFIC_RECRAWL_PATH)

def is_country_specific_entry(row):
    """
    Returns True if the entry's (timestp, id, cleaned_rootUrl) tuple exists
    in the corresponding country's set.
    """
    country = row['country']
    try:
        ts = int(row['timestp'])
    except Exception:
        return False
    entry_id = row['id']
    cleaned_rooturl = clean_url(row['rootUrl'])
    return (ts, entry_id, cleaned_rooturl) in country_specific_entries_set[country]

# Add the new column based on the combined matching condition
df_all['countrySpecificEntries'] = df_all.apply(is_country_specific_entry, axis=1)


In [25]:
# --------------------------------------------------------------------------------
# 3.1 Add "CrawlFailed", "CountriesCrawlFailed" columns
#
# "CrawlFailed": True if rootUrl is in this country's error logs 
#             
#
# "CountriesCrawlFailed": True if rootUrl appears in ANY country's error logs 
#                         (again ignoring InsecureCertificateError).
# 
# We already have:
#   - country_errors[country] = set of cleaned URLs that had an error != InsecureCertificateError
#   - all_failed_urls = global set of cleaned URLs that had an error in any country
# --------------------------------------------------------------------------------

df_all["CrawlFailed"] = False
df_all["CountriesCrawlFailed"] = False

# We assume the CSV has a column called "rootUrl". 
# If your column naming is different, adjust below.
if "rootUrl" not in df_all.columns:
    raise ValueError("Expected 'rootUrl' column not found in the entries CSVs.")

for idx, row in df_all.iterrows():
    c_country = row["country"]        # the crawler country
    root_url_raw = row["rootUrl"]     # the rootUrl from CSV
    c_root_url = clean_url(root_url_raw)
    
    # 1) Mark CrawlFailed if this country's error set contains the cleaned rootUrl
    if c_root_url in country_errors[c_country]:
        df_all.at[idx, "CrawlFailed"] = True
        
    # 2) Mark CountriesCrawlFailed if c_root_url in global all_failed_urls
    if c_root_url in all_failed_urls:
        df_all.at[idx, "CountriesCrawlFailed"] = True


In [26]:
# --------------------------------------------------------------------------------
# 3.2 Add "Potential Duplicates" column
#
# "Potential Duplicates": the number indicates the number of times that the site is
#                         crawled more than once. To determine the number of times
#                         a site is duplicated, we count the difference between the
#                         timestp for each site in each country, if the difference
#                         is more than 10 min, we consider it as a duplicated site.
# --------------------------------------------------------------------------------
threshold = 10 * 60_000

df_all['potentialDuplicates'] = 0

df_all['timestp'] = pd.to_numeric(df_all['timestp']).astype('Int64')
df_all.sort_values(['country', 'rootUrl', 'timestp'], inplace=True)
grouped_df = df_all.groupby(['country', 'rootUrl'])

for (country, root_url), group in grouped_df:
    potential_duplicate_count = 0
    prev_timestamp = None
    # Check each row in the group
    for index, row in group.iterrows():
        if prev_timestamp is None:
            prev_timestamp = row['timestp']
            continue
        time_diff = row['timestp'] - prev_timestamp
        if time_diff > threshold:
            # Increment potentialDuplicates for this row
            potential_duplicate_count += 1
        df_all.loc[index, 'potentialDuplicates'] = potential_duplicate_count
        prev_timestamp = row['timestp']
        

In [27]:
# --------------------------------------------------------------------------------
# 3.3 Add "Potential Rediction" column
# Note: this is identical to the Potential Duplicates column
#
# --------------------------------------------------------------------------------
threshold = 10 * 60_000

df_all['potentialRedirection'] = 0

df_all['timestp'] = pd.to_numeric(df_all['timestp']).astype('Int64')
df_all.sort_values(['country', 'rootUrl', 'timestp'], inplace=True)
grouped_df = df_all.groupby(['country', 'rootUrl'])

for (country, root_url), group in grouped_df:
    potential_duplicate_count = 0
    prev_timestamp = None
    # Check each row in the group
    for index, row in group.iterrows():
        if prev_timestamp is None:
            prev_timestamp = row['timestp']
            continue
        time_diff = row['timestp'] - prev_timestamp
        if time_diff > threshold:
            # Increment potentialRedirection for this row
            potential_duplicate_count += 1
        df_all.loc[index, 'potentialRedirection'] = potential_duplicate_count
        prev_timestamp = row['timestp']

In [28]:
# --------------------------------------------------------------------------------
# Create new column to fill the Url in
# --------------------------------------------------------------------------------
df_all['redirectionUrl'] = ""

In [29]:
# --------------------------------------------------------------------------------
# 4. add "parentCompanyNan" column
# Under the parentCompanyNan column, we will mark True if the parentCompany is nan, else False.
# --------------------------------------------------------------------------------

    
df_all['parentCompanyNan'] = df_all['parentCompany'].isna()

In [30]:
from urllib.parse import urlparse

def clean_url(url):
    """Cleans the URL, keeping only the essential domain.

    Args:
        url (str): The URL to clean.
        
    Returns:
        str: The cleaned domain.
    """
    parsed = urlparse(url)
    # Use netloc if present; if not, fall back to path (in case the scheme is missing)
    domain = parsed.netloc or parsed.path
    # Remove the "www." prefix if it exists
    if domain.startswith('www.'):
        domain = domain[4:]
    return domain

df_all['requestUrlCleaned'] = df_all['requestUrl'].apply(clean_url)


In [31]:
# --------------------------------------------------------------------------------
# 6. Reorder columns so that "country", "CrawlFailed", "CountriesCrawlFailed", and "httpRecrawl" are in the front.
# --------------------------------------------------------------------------------

all_cols = list(df_all.columns)
desired_start = ["country", "id", "CrawlFailed", "CountriesCrawlFailed", "httpRecrawl", "countrySpecificEntries", "potentialDuplicates", "potentialRedirection", "redirectionUrl", 'parentCompanyNan', "requestUrlCleaned"]
desired_order = (
    desired_start 
    + [c for c in all_cols if c not in desired_start]
)
df_all = df_all[desired_order]



In [32]:
# --------------------------------------------------------------------------------
# 7. Save the combined DataFrame to a final CSV file
# --------------------------------------------------------------------------------

output_file = MAIN_ENTRIES_PATH + "/combined_all_countries.csv"
df_all.to_csv(output_file, index=False, encoding="utf-8")
print(f"Final combined data saved to {output_file}.")

Final combined data saved to ../analysis_data/entries/combined_all_countries.csv.
